# 05 · Rules vs ML, and the locked final test

Notebook 04 showed the detector works across sites, but on that data the ML model agreed 100% with a simple rule ("any attack signal = attack"). A fair project must show what the ML adds. So here we compare three detectors:

| detector | what it is |
|---|---|
| **rules** | keyword patterns only, no learning (`eval/rules.py`) |
| **ML value** | XGBoost on `value_features()` |
| **ML value + TF-IDF** | same + character n-grams of what the user typed |

**Data**
- Training: **CSIC only** (normal + payload-carrying attacks). We tried adding half of Elsharawy's run 1 to training, and the models then scored 100% even on brute force, which no per-request model can really detect: they had learned the generator itself (its fixed fake password and target account), not attacks. So generated data is only used for testing.
- Development: all of run 1 + 60 benign requests whose text contains words like *select*, *drop*, *union*, *script* (`eval/benign_tricky.py`). Brute force is left out: one login attempt looks normal, so it belongs to the session layer.
- **Final test: run 2**, generated later with a new seed, disguised attacks and new payload lists. It is used once, in the last cell, after the code is frozen.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from ml.detector import build_pipeline, has_payload, to_frame
from eval.rules import rule_predict
from eval.benign_tricky import rows as benign_rows
from eval.match_labels import match

csic = pd.read_csv(ROOT / 'data' / 'processed' / 'csic_clean.csv', keep_default_na=False)
run1 = pd.read_csv(ROOT / 'data' / 'processed' / 'juiceshop_gen_labelled.csv', keep_default_na=False)
benign = benign_rows()
print(len(csic), 'CSIC,', len(run1), 'run1,', len(benign), 'benign tricky')

25608 CSIC, 3836 run1, 60 benign tricky


## 1. Training set (CSIC) and dev set (run 1 + benign tricky)

In [2]:
COLS = ['method', 'path', 'body', 'label', 'attack_type']
dev = pd.concat([run1[run1.attack_type != 'bruteforce'][COLS], benign], ignore_index=True)

y_csic = csic['label'].values
X_csic = to_frame(csic['Method'], csic['path'], csic['body'])
keep = (y_csic == 0) | has_payload(X_csic).values
X_train, y_train = X_csic[keep], y_csic[keep]
HAND = [c for c in X_train.columns if c != 'text']
POS_WEIGHT = (y_train == 0).sum() / (y_train == 1).sum()
print(f'train: {len(y_train)} CSIC requests ({y_train.sum()} attacks)')
print(f'dev:   {len(dev)} requests ({(dev.label != "normal").sum()} attacks)')

train: 15428 CSIC requests (5784 attacks)
dev:   3699 requests (381 attacks)


## 2. Compare the three detectors on the dev set

In [3]:
def evaluate(df, models):
    y = (df['label'] != 'normal').astype(int).values
    X = to_frame(df['method'], df['path'], df['body'])
    preds = {'rules': rule_predict(df['method'], df['path'], df['body'])}
    for name, m in models.items():
        preds[name] = m.predict(X)
    table = pd.DataFrame({name: {
        'precision': precision_score(y, p, zero_division=0),
        'recall': recall_score(y, p, zero_division=0),
        'F1': f1_score(y, p, zero_division=0),
        'false alarms': int(((p == 1) & (y == 0)).sum()),
        'missed attacks': int(((p == 0) & (y == 1)).sum()),
    } for name, p in preds.items()}).T
    return table, preds

models = {
    'ML value': build_pipeline(HAND, pos_weight=POS_WEIGHT).fit(X_train, y_train),
    'ML value + TF-IDF': build_pipeline(HAND, pos_weight=POS_WEIGHT, use_tfidf=True).fit(X_train, y_train),
}
dev_table, dev_preds = evaluate(dev, models)
dev_table.round(3)

,precision,recall,F1,false alarms,missed attacks
rules,1.0,0.745,0.854,0.0,97.0
ML value,1.0,1.000,1.000,0.0,0.0
ML value + TF-IDF,1.0,0.992,0.996,0.0,3.0


## 3. Why the rules miss attacks

Recall by attack type on the dev set.

In [4]:
is_atk = (dev.label != 'normal').values
atk = dev[is_atk].assign(**{name: p[is_atk] for name, p in dev_preds.items()})
atk.groupby('attack_type')[list(dev_preds)].mean().round(2)

,rules,ML value,ML value + TF-IDF
attack_type,,,
sqli,0.89,1.0,0.98
traversal,0.33,1.0,1.00
xss,1.00,1.0,1.00


- **Traversal:** most attempts are double-encoded, so the exact `../` pattern never appears. The ML learns that double encoding itself is a strong signal.
- **SQLi:** some variants do not match any exact pattern; the ML still catches them from the combination of signals.

## 4. What the ML still gets wrong

Normal dev requests the chosen model calls attacks (all text is made up in the lab).

In [5]:
chosen = dev_preds['ML value']
dev[(chosen == 1) & (dev.label == 'normal')][['method', 'path', 'body']]

,method,path,body


**Decision:** TF-IDF does not help here (it misses a few SQLi), so the backend keeps **ML value** (`ml/detector.py`, no TF-IDF).

**Tuning note:** an earlier version flagged the benign comment "buy one get one, 1=1 deal" (2 false alarms). We changed `v_tautology` so `1=1` only counts right after SQL syntax (OR, AND, WHERE, a quote). This was tuned on the dev set, so the dev score is optimistic; run 2 below is the honest number.

---
## 5. LOCKED FINAL TEST (run 2)

**Rules for this cell:**
1. Run it once, only after all code above (and in `ml/`) is frozen and committed.
2. Do not change any feature, model or threshold after seeing its output. If something must change, the change and the reason go in the report, and we need a new unseen run.

Same models as above (trained on CSIC only); run 2 is only used for testing. Brute force is reported by the session layer, not here.

In [6]:
RUN_FINAL = True   # set to True once, when the code is frozen

if RUN_FINAL:
    run2, report = match(ROOT / 'data' / 'raw' / 'juiceshop_generated_run2.log',
                         ROOT / 'data' / 'labels' / 'requests_run2.csv')
    print('matching:', report)
    test = run2[run2.attack_type != 'bruteforce'][COLS]
    final_table, final_preds = evaluate(test, models)
    final_table.to_csv(ROOT / 'data' / 'processed' / 'final_test_run2.csv')
    display(final_table.round(3))
else:
    print('Final test not run yet (RUN_FINAL = False).')

matching: {'log_lines': 4368, 'labels': 4368, 'matched': 4368, 'unmatched_log_lines': 0, 'unmatched_labels': 0}


,precision,recall,F1,false alarms,missed attacks
rules,1.0,0.623,0.767,0.0,140.0
ML value,1.0,0.987,0.993,0.0,5.0
ML value + TF-IDF,1.0,0.970,0.985,0.0,11.0
